In [207]:
# load the data using pandas
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)
df = pd.read_csv("SubmittedQuestionResponses-202209061924.csv")

In [208]:
'''
Logic:
since anomalies are assumed to be small in number, for each question_id,
    1) for each type of response we can have: total_score_response , count_response
    2) we can keep track of the possible scores for each question_id
    3) the true score for each response must be then which ever possible score is closest to total_score_response / count_response
    4) then, we can compare the correct answer with all the responses and see whether the scores match.
    5) If not, then we can add it to an anomalies dataframe (taqr_id, question_id, response,answer,true_score).

but you can have partial grades,
'''

'\nLogic:\nsince anomalies are assumed to be small in number, for each question_id,\n    1) for each type of response we can have: total_score_response , count_response\n    2) we can keep track of the possible scores for each question_id\n    3) the true score for each response must be then which ever possible score is closest to total_score_response / count_response\n    4) then, we can compare the correct answer with all the responses and see whether the scores match.\n    5) If not, then we can add it to an anomalies dataframe (taqr_id, question_id, response,answer,true_score).\n\nbut you can have partial grades,\n'

In [209]:
'''Note: The question_id is sorted in ascending order'''
# get all the different question_ids and store them in a list
question_ids = df["question_id"].unique().tolist() # works

q_r_count = {}
# now for each question, get the unique responses and map it to a dictionary
responses_by_qid = df.groupby("question_id")["response"].unique().apply(list)

# responses_by_qid shows that for certain question_ids, there are responses which shouldn't even be there
# for example if many reponses have A, B, C or D as answers, another response can't be 1=>(EMPTY), ....
# thus it is necessary to do counts of which responses appear how many times.
# And then from a case by case basis, looking at the statistics, some responses must be called illegitimate.

# next, see what's the total score for each possible response and their count
result = (
    df.groupby(["question_id", "response"])
      .agg(
          total_score=("score", "sum"),
          count=("score", "size")
      )
      .reset_index()
)

# gives the unique possible scores for each question_id
unique_scores = df.groupby("question_id")["score"].unique()
# this showed me that for question_id = 12563, one of the scores is negative.
'''so clearly, all negative scores represent anomalies assuming no negative marking'''



'so clearly, all negative scores represent anomalies assuming no negative marking'

## question_id = 12563

In [210]:
# Working on question_id = 12563
filtered_df = result[result["question_id"] == 12563]

res_df = df[df["question_id"] == 12563] # only question_id = 12563
# anomalies_1 dataframe ("taqr_id","question_id","response","score")
rows = []
for row in res_df.itertuples(index=False):
    # false negative condition:
    if (row.response == "A" and row.score != 1):
        # since A is correct, "A" should be given a score of 1
        sc = 1

    # false positive condition
    elif (row.response != "A" and row.score != 0) :
        # any other answer should be given 0
        sc = 0
    else:
        continue

    rows.append({
        "taqr_id":row.taqr_id,
        "question_id": row.question_id,
        "response": row.response,
        "Original score": row.score,
        "New score": sc
    })

anaomalies_1 = pd.DataFrame(rows)

## Question_id = 12595

In [211]:
filtered_df = result[result["question_id"] == 12595]
res_df = df[df["question_id"] == 12595] # only question_id = 12563

# true answers list
true_ans = ['borrow=>[Receiving money with an agreement to repay it in the future, usually with interest charged.',
 'lend=>[Giving money to a person or a business with the expectation that it will be repaid.',
 'trade=>[The transfer of goods and services, often in exchange for money from one individual or organization to another.]',
 "echanges=>[Transférer des biens et des services, souvent en échange d'argent, d'un individu à un autre ou d'une organisation à une autre.",
 "emprunt=>[Recevoir de l'argent en ayant pour accord de le rembourser plus tard, généralement avec des intérêts.",
 "pret=>[Donner de l'argent à une personne ou à une entreprise en s'attendant à ce qu'il soit remboursé.]"]

# Now, since each correct option carries 1/3 of the grade,
# I can create a helper function to see how many sub responses match to those in the true answers list and calculat the true score.

def help_true_scorer_1(response):
    '''returns the true score for this response'''
    subparts = response.split("], ")
    score = 0
    for part in subparts:
        if part in true_ans:
            score +=2/3
    return score

rows = []
for row in res_df.itertuples(index=False):

    true_score = np.round(help_true_scorer_1(str(row.response)),3)

    if np.round(row.score,3) != true_score:
        rows.append({
            "taqr_id":row.taqr_id,
            "question_id": row.question_id,
            "response": row.response,
            "Original score": row.score,
            "New score": true_score
        })

anaomalies_2 = pd.DataFrame(rows)


## Question_id = 17404

In [212]:
filtered_df = result[result["question_id"] == 17404]
filtered_df['total_score / count'] = filtered_df['total_score'] / filtered_df['count']
filtered_df = filtered_df.sort_values('count',ascending=False)
res_df = df[df["question_id"] == 17404]

true_ans =['1=>(EMPTY)',
            ' 1=>2',
            ' 2=>(EMPTY)',
            ' 2=>5',
            ' 3=>3',
            ' 3=>1',
            ' 4=>4',
            ' 5=>(EMPTY)',
            ' 6=>6']

def help_true_scorer_2(response):
    '''returns the true score for this response'''
    subparts = response.split(",")
    score = 0
    for part in subparts:
        if part in true_ans:
            score +=2/9
    return score

rows = []
for row in res_df.itertuples(index=False):

    true_score = np.round(help_true_scorer_2(str(row.response)),3)

    if np.round(row.score,3) != true_score:
        rows.append({
            "taqr_id":row.taqr_id,
            "question_id": row.question_id,
            "response": row.response,
            "Original score": row.score,
            "New score": true_score
        })
anaomalies_3 = pd.DataFrame(rows)
# filtered_df.iloc[:10]

/var/folders/qy/_rg_prqx1kn6x4b6bg336mk40000gn/T/ipykernel_6202/758084579.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['total_score / count'] = filtered_df['total_score'] / filtered_df['count']


## question_id = 17495

In [213]:
filtered_df = result[result["question_id"] == 17495]
filtered_df['total_score / count'] = filtered_df['total_score'] / filtered_df['count']
filtered_df = filtered_df.sort_values('count',ascending=False)
res_df = df[df["question_id"] == 17495] # only question_id = 17495
true_ans = ['1=>1', # max score is 2,
        ' 1=>5',    # so each subpart is 2/10
        ' 2=>2',
        ' 2=>1',
        ' 3=>3',
        ' 3=>1',
        ' 4=>4',
        ' 4=>6',
        ' 5=>5',
        ' 6=>6']

def help_true_scorer_3(response):
    '''returns the true score for this response'''
    subparts = response.split(",")
    score = 0
    for part in subparts:
        if part in true_ans:
            score +=2/10
    return score

rows = []
for row in res_df.itertuples(index=False):

    true_score = np.round(help_true_scorer_3(str(row.response)),3)

    if np.round(row.score,3) != true_score:
        rows.append({
            "taqr_id":row.taqr_id,
            "question_id": row.question_id,
            "response": row.response,
            "Original score": row.score,
            "New score": true_score
        })
anaomalies_4 = pd.DataFrame(rows)

/var/folders/qy/_rg_prqx1kn6x4b6bg336mk40000gn/T/ipykernel_6202/3418235744.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['total_score / count'] = filtered_df['total_score'] / filtered_df['count']


In [214]:
'''Single Solution'''
'''
    question_id : [
                score per subpart,
                [..list of correct subparts...]
                ]
    '''
q_dic = {
         12563:[1, # max score is 1, so each subpart is 1 because only one option to choose
                ['A']],
         12595:[2/3, # max score is 2, so each subpart is 2/3 because each response is only in one language (only 3 subparts)
                    ['borrow=>[Receiving money with an agreement to repay it in the future, usually with interest charged.',
                    'lend=>[Giving money to a person or a business with the expectation that it will be repaid.',
                    'trade=>[The transfer of goods and services, often in exchange for money from one individual or organization to another.]',
                    "echanges=>[Transférer des biens et des services, souvent en échange d'argent, d'un individu à un autre ou d'une organisation à une autre.",
                    "emprunt=>[Recevoir de l'argent en ayant pour accord de le rembourser plus tard, généralement avec des intérêts.",
                    "pret=>[Donner de l'argent à une personne ou à une entreprise en s'attendant à ce qu'il soit remboursé.]"]],
         17404:[2/9,['1=>(EMPTY)', # max score is 2,
                    ' 1=>2',       # so each subpart is 2/9
                    ' 2=>(EMPTY)',
                    ' 2=>5',
                    ' 3=>3',
                    ' 3=>1',
                    ' 4=>4',
                    ' 5=>(EMPTY)',
                    ' 6=>6']],
         17495:[2/10,['1=>1', # max score is 2,
                    ' 1=>5',  # so each subpart is 2/10
                    ' 2=>2',
                    ' 2=>1',
                    ' 3=>3',
                    ' 3=>1',
                    ' 4=>4',
                    ' 4=>6',
                    ' 5=>5',
                    ' 6=>6']]
        }

def true_scorer(response,grade,true_ans,splt):
    '''
        returns the true score for this response
        grade: score for each subpart of response
        true_ans: the list of true subpart answers as in q_dic
    '''
    subparts = response.split(splt)
    score = 0
    for part in subparts:
        if part in true_ans:
            score +=grade
    return score # the grade which should be given for this response.

rows = []
for question_id in q_dic.keys():

    # responses to question_id = 12595 have commas inside each subpart
    # so they must be separated using other splitting string splt i.e. "], "
    # Others are separated by commas: ","
    if question_id == 12595:
        splt = "], "
    else:
        splt = ","

    res_df = df[df["question_id"] == question_id] # filtering responses of one particular question_id

    '''total count'''
    a = res_df.groupby(['question_id','response']).agg( total_score=("score", "sum"),total_count=("score", "size"),
                                                       ).reset_index().sort_values(by="total_count", ascending=False)
    a["ratio"] = a["total_score"] / a["total_count"]
    display(a.head())
    for row in res_df.itertuples(index=False):

        true_score = np.round(true_scorer(str(row.response), q_dic[question_id][0],q_dic[question_id][1], splt),3) # rounding to two decimal places

        if np.round(row.score,3) != true_score:
            rows.append({
                "taqr_id":row.taqr_id,
                "question_id": row.question_id,
                "response": row.response,
                "Original score": row.score,
                "New score": true_score
            })

anomalies = pd.DataFrame(rows) # have to put this into a csv


,question_id,response,total_score,total_count,ratio
7,12563,D,0.0,24282,0.000000
6,12563,C,2.0,20960,0.000095
2,12563,A,16913.0,16914,0.999941
4,12563,B,6.0,16496,0.000364
0,12563,"1=>(EMPTY), 1=>3, 2=>(EMPTY), 2=>2, 3=>(EMPTY), 3=>1, 4=>(EMPTY), 4=>4",1.0,1,1.000000


,question_id,response,total_score,total_count,ratio
18,12595,"borrow=>[Receiving money with an agreement to repay it in the future, usually with interest charged.], lend=>[Giving money to a person or a business with the expectation that it will be repaid.], trade=>[The transfer of goods and services, often in exchange for money from one individual or organization to another.]",49330.000001,24804,1.988792
13,12595,"borrow=>[Giving money to a person or a business with the expectation that it will be repaid.], lend=>[Receiving money with an agreement to repay it in the future, usually with interest charged.], trade=>[The transfer of goods and services, often in exchange for money from one individual or organization to another.]",4703.335685,7088,0.663563
20,12595,"borrow=>[Receiving money with an agreement to repay it in the future, usually with interest charged.], lend=>[The transfer of goods and services, often in exchange for money from one individual or organization to another.], trade=>[Giving money to a person or a business with the expectation that it will be repaid.]",1447.334057,2192,0.660280
37,12595,"echanges=>[Transférer des biens et des services, souvent en échange d'argent, d'un individu à un autre ou d'une organisation à une autre.], emprunt=>[Recevoir de l'argent en ayant pour accord de le rembourser plus tard, généralement avec des intérêts.], pret=>[Donner de l'argent à une personne ou à une entreprise en s'attendant à ce qu'il soit remboursé.]",2564.000000,1293,1.982985
22,12595,"borrow=>[The transfer of goods and services, often in exchange for money from one individual or organization to another.], lend=>[Giving money to a person or a business with the expectation that it will be repaid.], trade=>[Receiving money with an agreement to repay it in the future, usually with interest charged.]",670.667002,1016,0.660105


,question_id,response,total_score,total_count,ratio
33,17404,"1=>(EMPTY), 1=>2, 2=>(EMPTY), 2=>5, 3=>3, 3=>1, 4=>4, 5=>(EMPTY), 6=>6",25101.333330,12551,1.999947
8,17404,"1=>(EMPTY), 1=>1, 2=>(EMPTY), 2=>3, 3=>(EMPTY), 3=>2, 4=>4, 5=>5, 6=>6",0.000000,4059,0.000000
10,17404,"1=>(EMPTY), 1=>1, 2=>(EMPTY), 2=>5, 3=>3, 3=>2, 4=>4, 5=>(EMPTY), 6=>6",2468.001234,3702,0.666667
31,17404,"1=>(EMPTY), 1=>2, 2=>(EMPTY), 2=>3, 3=>(EMPTY), 3=>1, 4=>4, 5=>5, 6=>6",4481.322130,3361,1.333330
9,17404,"1=>(EMPTY), 1=>1, 2=>(EMPTY), 2=>4, 3=>3, 3=>2, 4=>(EMPTY), 5=>5, 6=>6",0.000000,680,0.000000


,question_id,response,total_score,total_count,ratio
245,17495,"1=>1, 1=>5, 2=>2, 2=>1, 3=>3, 3=>3, 4=>4, 4=>6, 5=>5, 6=>6",22303.5,14869,1.500000
234,17495,"1=>1, 1=>5, 2=>2, 2=>1, 3=>3, 3=>1, 4=>4, 4=>6, 5=>5, 6=>6",23412.0,11707,1.999829
239,17495,"1=>1, 1=>5, 2=>2, 2=>1, 3=>3, 3=>2, 4=>4, 4=>6, 5=>5, 6=>6",13354.5,8903,1.500000
275,17495,"1=>1, 1=>5, 2=>2, 2=>3, 3=>3, 3=>1, 4=>4, 4=>6, 5=>5, 6=>6",4830.0,3220,1.500000
282,17495,"1=>1, 1=>5, 2=>2, 2=>3, 3=>3, 3=>3, 4=>4, 4=>6, 5=>5, 6=>6",2477.0,2477,1.000000
